# 🔺 Delta Lake — Change Data Feed (CDF)

---

## 1. O que é o Change Data Feed?

O `_delta_log` já rastreia mudanças no nível de **arquivo** (quais Parquets foram adicionados ou removidos).  
O **CDF** vai um nível abaixo: rastreia mudanças no nível de **linha** — qual linha foi inserida, atualizada ou deletada.

```
  _delta_log (nível de arquivo)        CDF (nível de linha)
  ─────────────────────────────        ─────────────────────────────
  versão 3 → add: part-002.parquet     versão 3 → INSERT id=4 name='Rock'
             remove: part-001.parquet             INSERT id=5 name='Wolf'
                                                  UPDATE id=3 (antes: 'Bat', depois: 'Tiger')
                                                  DELETE id=1
```

### Casos de uso

| Caso de uso | Como o CDF ajuda |
|---|---|
| **Processamento incremental** | Consumidores downstream processam apenas as linhas que mudaram, sem reprocessar a tabela inteira |
| **Audit log detalhado** | Rastreia exatamente o que mudou, quando e como (antes e depois do UPDATE) |
| **Otimização de pipelines** | Evita joins e scans completos — lê apenas o delta de mudanças |
| **Streaming de deleções** | Captura linhas deletadas em tempo real e as envia para outros sistemas |

---

### Como o CDF funciona internamente?

Quando o CDF está habilitado, o Delta Lake cria um subdiretório `_change_data/` dentro da tabela.  
Cada operação que modifica linhas gera um Parquet nesse diretório com as linhas afetadas e uma coluna extra `_change_type`.

```
  demo/
  ├── _delta_log/
  ├── part-001.snappy.parquet   (dados atuais)
  └── _change_data/
        └── part-cdf-001.snappy.parquet   ← linhas modificadas + _change_type
```

O campo `_change_type` pode ter os seguintes valores:

| Valor | Quando aparece |
|---|---|
| `insert` | Linha inserida via `INSERT` |
| `update_preimage` | Estado da linha **antes** do `UPDATE` |
| `update_postimage` | Estado da linha **depois** do `UPDATE` |
| `delete` | Linha removida via `DELETE` |

---

## 2. Habilitando CDF para todas as novas tabelas

Esta configuração de sessão faz com que qualquer nova tabela criada já tenha o CDF ativado por padrão.

In [ ]:
%%sql
-- Habilita CDF globalmente para novas tabelas na sessão atual
SET spark.databricks.delta.properties.defaults.enableChangeDataFeed = true;

---
## 3. Criando a tabela demo

Criamos a tabela e inserimos dados iniciais **antes** de habilitar o CDF na tabela.  
Isso é intencional — o CDF só rastreia mudanças a partir do momento em que é ativado.

In [ ]:
%%sql
DROP TABLE IF EXISTS demo;

CREATE TABLE demo (
  id    INT,
  name  STRING
);

INSERT INTO demo VALUES 
(1, 'Miki'),
(2, 'Spider'),
(3, 'Bat');

In [ ]:
-- Estrutura sem CDF: apenas Parquets e _delta_log, sem _change_data/
%fs
ls dbfs:/user/hive/warehouse/demo

---
## 4. Habilitando CDF na tabela e realizando operações

O CDF pode ser habilitado em uma tabela existente via `TBLPROPERTIES`.  
A partir desse momento, toda operação DML que modifique linhas será rastreada no `_change_data/`.

In [ ]:
%%sql
-- Habilita CDF apenas para esta tabela
ALTER TABLE demo SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [ ]:
%%sql
-- INSERT: rastreado como 'insert' no CDF
INSERT INTO demo VALUES 
(4, 'Rock'),
(5, 'Wolf');

-- UPDATE: rastreado como 'update_preimage' e 'update_postimage' no CDF
UPDATE demo
SET name = 'Tiger'
WHERE id = 3;

-- DELETE: rastreado como 'delete' no CDF
DELETE FROM demo
WHERE id = 1;

### Inspecionando o diretório após as operações

In [ ]:
-- O subdiretório _change_data/ agora aparece no diretório da tabela
%fs
ls dbfs:/user/hive/warehouse/demo

In [ ]:
-- Dentro de _change_data/ há um Parquet com todas as linhas modificadas
%fs
ls dbfs:/user/hive/warehouse/demo/_change_data/

In [ ]:
%%sql
-- Versão 2 corresponde à ativação do CDF (ALTER TABLE SET TBLPROPERTIES)
-- A partir dessa versão, as mudanças passam a ser rastreadas
DESC HISTORY demo;

---
## 5. Consultando as mudanças com `table_changes`

A função `table_changes` retorna todas as mudanças a partir de uma versão ou timestamp,  
adicionando automaticamente as colunas de metadados do CDF a cada linha.

In [ ]:
%%sql
-- Lê todas as mudanças a partir da versão 2 (quando o CDF foi ativado)
-- Colunas extras: _change_type, _commit_version, _commit_timestamp
SELECT * FROM table_changes('demo', 2);

> **Lendo o resultado:**
>
> | `_change_type` | O que representa |
> |---|---|
> | `insert` | Linhas novas: id=4 (Rock) e id=5 (Wolf) |
> | `update_preimage` | Estado do id=3 **antes** do UPDATE: name='Bat' |
> | `update_postimage` | Estado do id=3 **depois** do UPDATE: name='Tiger' |
> | `delete` | Linha removida: id=1 (Miki) |
>
> Note que o UPDATE gera **duas entradas** por linha modificada — o antes e o depois.  
> Isso permite auditar exatamente o que mudou em cada campo.

---

## 6. Caso de uso — Streaming de deleções

Cenário: precisamos enviar em "tempo real" todas as linhas deletadas da tabela `demo`  
para outra tabela `demo_deleted_data`, que será consumida por outro time.

```
  demo  ──(CDF stream, filter: delete)──►  demo_deleted_data
```

O Spark Structured Streaming com `readChangeFeed` monitora continuamente o `_change_data/`  
e processa apenas as novas entradas — sem reprocessar dados já vistos.

In [ ]:
%%sql
DROP TABLE IF EXISTS demo_deleted_data;

CREATE TABLE demo_deleted_data (id INT, name STRING);

In [ ]:
%python
# readChangeFeed=true → lê do _change_data/ em vez dos Parquets principais
# filter _change_type = 'delete' → captura apenas linhas deletadas
# writeStream append → acumula as deleções na tabela de destino
spark.readStream.format('delta')\
	  .option('readChangeFeed','true')\
	  .table('demo')\
	  .filter("_change_type = 'delete'")\
	  .select('id','name')\
	.writeStream\
	  .outputMode('append')\
	  .option('checkpointLocation','/FileStore/checkpointDirCD')\
	  .table('demo_deleted_data')

In [ ]:
%%sql
-- id=1 (Miki) aparece aqui — foi deletado anteriormente e capturado pelo stream
SELECT * FROM demo_deleted_data;

### Testando o stream com uma nova deleção

In [ ]:
%%sql
-- Deleta id=3 (Tiger) — o stream deve capturar essa mudança automaticamente
DELETE FROM demo
WHERE id = 3;

In [ ]:
%%sql
-- Confirma a nova deleção no CDF da tabela source
SELECT * FROM table_changes('demo', 2);

In [ ]:
%%sql
-- id=1 (Miki) e id=3 (Tiger) agora aparecem na tabela de destino
SELECT * FROM demo_deleted_data;

> **Como o stream sabe o que já processou?**  
> O `checkpointLocation` armazena o progresso do stream (última versão lida).  
> Na próxima execução, o stream continua de onde parou — sem reprocessar deleções antigas.

---

## 7. Outras formas de consultar o CDF

A função `table_changes` aceita versão ou timestamp como parâmetros, e também suporta ranges.

In [ ]:
%%sql
-- Por timestamp exato (use o valor da coluna 'timestamp' do DESC HISTORY)
SELECT * FROM table_changes('demo', 'Timestamp');

In [ ]:
%%sql
-- Por range de timestamps: todas as mudanças entre Timestamp1 e Timestamp2
SELECT * FROM table_changes('demo', 'Timestamp1', 'Timestamp2');

In [ ]:
%%sql
-- Por range de versões: todas as mudanças entre as versões 2 e 4 (inclusive)
SELECT * FROM table_changes('demo', 2, 4);

> **Dica:** ao usar ranges, o CDF retorna apenas as mudanças dentro da janela especificada.  
> Isso é ideal para pipelines batch que processam incrementalmente um intervalo de tempo ou versão.

---

## 8. Resumo — Change Data Feed

```
┌───────────────────────────────────────────────────────────────────┐
│                  CHANGE DATA FEED — FLUXO                         │
│                                                                   │
│  DML (INSERT / UPDATE / DELETE / MERGE)                           │
│         │                                                         │
│         ├──► _delta_log  (rastreia arquivos)    ← sempre ativo    │
│         │                                                         │
│         └──► _change_data/  (rastreia linhas)  ← apenas com CDF  │
│                    │                                              │
│                    └──► table_changes('demo', versão)             │
│                              │                                    │
│                    ┌─────────┼──────────┐                         │
│                    │         │          │                         │
│                 insert   update_*   delete                        │
│                              │                                    │
│                    preimage / postimage                           │
│                    (antes e depois do UPDATE)                     │
└───────────────────────────────────────────────────────────────────┘
```

### Comandos de referência

| Objetivo | Comando |
|---|---|
| Habilitar CDF globalmente | `SET spark.databricks.delta.properties.defaults.enableChangeDataFeed = true` |
| Habilitar CDF em tabela existente | `ALTER TABLE demo SET TBLPROPERTIES (delta.enableChangeDataFeed = true)` |
| Consultar mudanças por versão | `SELECT * FROM table_changes('demo', 2)` |
| Consultar mudanças por timestamp | `SELECT * FROM table_changes('demo', 'Timestamp')` |
| Consultar range de versões | `SELECT * FROM table_changes('demo', 2, 4)` |
| Consultar range de timestamps | `SELECT * FROM table_changes('demo', 'T1', 'T2')` |
| Stream de mudanças | `.option('readChangeFeed', 'true')` no `readStream` |